# 👷 Projet 11 — Détection d'EPI sur photos de chantier (YOLO + Gradio)

Uploade une photo de chantier → le modèle détecte les **casques** et te dit qui n'en porte pas.

**Avant de lancer :** `Exécution` → `Modifier le type d'exécution` → **GPU (T4)**.
Puis `Exécution` → `Tout exécuter`. Un **lien public Gradio** apparaît à la fin — clique dessus.

## 1. Installer les librairies

In [ ]:
!pip install -q ultralyticsplus==0.0.29 ultralytics==8.0.43 gradio
print('✓ Installation terminée')

## 2. Charger le modèle PPE pré-entraîné
Modèle `keremberke/yolov8m-hard-hat-detection` : détecte casques / têtes sans casque.

In [ ]:
from ultralyticsplus import YOLO, render_result

model = YOLO('keremberke/yolov8m-hard-hat-detection')
model.overrides['conf'] = 0.25   # seuil de confiance minimum
model.overrides['iou'] = 0.45    # seuil de recouvrement des boîtes
model.overrides['max_det'] = 1000

print('✓ Modèle chargé')
print('Classes détectées :', model.model.names)

## 3. Fonction de détection

In [ ]:
def detecter(image):
    """Prend une image, renvoie l'image annotée + un résumé texte."""
    results = model.predict(image)
    image_annotee = render_result(model=model, image=image, result=results[0])

    # Compter les objets détectés par classe
    noms = model.model.names
    compte = {}
    for c in results[0].boxes.cls:
        nom = noms[int(c)]
        compte[nom] = compte.get(nom, 0) + 1

    if compte:
        resume = '\n'.join(f'• {nom} : {n}' for nom, n in compte.items())
    else:
        resume = 'Aucun objet détecté (essaie une autre photo ou baisse le seuil)'
    return image_annotee, resume

print('✓ Fonction prête')

## 4. Lancer l'interface Gradio 🚀

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=detecter,
    inputs=gr.Image(type='pil', label='📷 Photo de chantier'),
    outputs=[
        gr.Image(type='pil', label='Résultat annoté'),
        gr.Textbox(label='Résumé de la détection'),
    ],
    title="👷 Détection d'EPI sur chantier",
    description="Glisse une photo. Le modèle détecte les casques (Hardhat) et les têtes sans casque (NO-Hardhat).",
)

demo.launch(share=True)  # share=True → lien public cliquable